# Example of Photometry

For each MCMC sample, the corresponding profile parameters can be input into Lenstronomy's `image_linear_solve` function to determine the optimal fluxes of all light components (e.g., lens light, source light, and quasar images if applicable). To obtain morphological properties for multi-component lens light models, an instance of Lenstronomy's `LightProfileAnalysis` class is instantiated.

# Setting up the Arguments

The `Photometry` class requires the following inputs:

1) An instance of the `Output` class
2) The system name corresponding to the model in which the linear inversion will be evaluated.
3) The model ID of the run to optimize.
4) A dictionary mapping the data filters and desired model components to utilize in the linear inversion. For the lensed_quasar example, we have 1 lens light profile, 1 source light profile, and 4 point sources to optimize. `Photometry` already handles point sources, so we only need to specify the indices of the lens light and source light components that we want to keep in the optimization, per filter, as seen below. If we also had, for example, a UNIFORM lens light profile, then we would exclude that from the lens light flux measurement via `exclude_lens_light_indices: [1]`. But note, the index would also need to be included in `lens_light_indices` so that the Linear Solver reconstructs the proper model configuration: `lens_light_indices: [0, 1]`.
5) The walker ratio used in the MCMC process.
6) *optional:* The number of burn-in iterations to discard from the MCMC.
7) *optional:* The type of aperture in which the linear inversion is evaluated. Current options are "circle" and "square." If `None`, the inversion will be calculated over the full image grid.
8) *optional:* The size in arcseconds of the aperture in which the inversion is to be evaluated.
9) *optional:* **do_morphology**: Boolean that dictates whether or not morphological properties of multi-component lens light profiles are evaluated.

# Imports

In [4]:
from dolphin.analysis.output import Output
from dolphin.analysis.photometry import Photometry
import numpy as np

# create Output instance

In [5]:
output = Output("../io_directory_example/")

# create band_config dictionary

In [6]:
band_config = {
    "F814W": {
        "lens_light_indices": [0],
        "source_indices": [0],
        "exclude_lens_light_indices": [],
    }
}

# If we had multiple filters, it would look like so:

# band_config = {
#    "F814W": {
#        "lens_light_indices": [0],
#        "source_indices": [0],
#        "exclude_lens_light_indices": [],
#    },
#    "F475X": {
#        "lens_light_indices": [1],
#        "source_indices": [1],
#        "exclude_lens_light_indices": [],
#    }
# }

# create Photometry instance

For this example, we will only use the last 25 steps of the MCMC. We will also not use a circular/square aperture, and instead evaluate over the full image grid. Even though this system uses one lens light model, we will still solve for the light morphology for the purposes of this example.

In [7]:
system_name = "lensed_quasar"
model_id = "example"

phot = Photometry(
    output,
    lens_name=system_name,
    model_id=model_id,
    band_config=band_config,
    walker_ratio=2,
    burn_in=-25,
    aperture_type=None, # options: "circle", "square", None
    aperture_size=None,
    do_morphology=True,
)

Loaded output for lensed_quasar with model ID example.
dolphin version used: 1.4.0
lenstronomy version used: 1.13.6


# perform linear inversion, convert to magnitudes, and save outputs

By default, Lenstronomy's linear inversion returns fluxes in units in which the data is taken. As such, one can convert these to magnitudes by hand or use the helper functions implemented in Dolphin. Currently, the following instruments are supported (alongside the required calibration parameters from their FITS headers):

1) JWST:
- `pixar_sr`: JWST average pixel area in units of steradians

2) HST:
- `photflam`: mean flux density (in erg cm–2 sec–1 Å–1) that produces 1 count per second in the HST observing mode
- `photzpt`: STMAG HST data band zero point
- `photplam`: HST filter pivot wavelength

The fluxes are converted to AB magnitudes as follows:

# 1) For JWST data

For JWST images, the pixel flux is converted into Janskys using the average pixel area:

$$
F_{\rm Jy}
=
F
\times
\mathrm{PIXAR\_SR}
\times
10^6,
$$

The AB magnitude is then computed directly from the standard definition:

$$
m_{\rm AB}
=
-2.5\log_{10}
\left(
\frac{F_{\rm Jy}}{3631}
\right),
$$

where 3631 Jy is the AB magnitude zero point flux density.

# 2) For HST data

The pixel flux returned by the linear inversion is first converted into physical flux density units:

$$
F_\lambda = F \times \mathrm{PHOTFLAM},
$$

The corresponding ST magnitude is then computed as

$$
m_{\rm ST}
=
-2.5\log_{10}(F_\lambda)
- \rm PHOTZPT,
$$


Finally, the ST magnitude is converted into an AB magnitude:

$$
m_{\rm AB}
=
m_{\rm ST}
-
5\log_{10}(\mathrm{PHOTPLAM})
+
2.5\log_{10}(c)
-
27.5,
$$

To convert the output fluxes to magnitudes, one must pass a dictionary that maps the data filters to their respective calibration parameters. An example is given below.

In [ ]:
flux_results, morphology_results = phot.do_linear_inversion()

calibration_parameters = {
    "F814W": {"instrument": "HST", "photflam": 1.52122335e-19, "photzpt": -21.1, "photplam": 8034.189}
}

magnitude_results = phot.calculate_ab_magnitude(
    flux_chain=flux_results, calibration_parameters=calibration_parameters
)

phot.save_to_hdf5(flux_results, magnitude_results, morphology_results)

Loaded output for lensed_quasar with model ID example.
dolphin version used: 1.4.0
lenstronomy version used: 1.13.6


# Load the outputs and analyze some statistics

In [9]:
flux_chain = phot.load_flux_chain()
mag_chain = phot.load_magnitude_chain()
morph_chain = phot.load_morphology_chain()

# Image 1
print(f"Image 1 Average Magnitude: {np.mean(mag_chain["F814W"]["image1"])}")

# Lens
print(f"Lens Average Magnitude: {np.mean(mag_chain["F814W"]["lens"])}")

# Extended Source
print(f"Extended Source Average Magnitude: {np.mean(mag_chain["F814W"]["source_lensed"])}")

# Instrinsic Source
print(f"Intrinsic Source Average Magnitude: {np.mean(mag_chain["F814W"]["source_intrinsic"])}")

Image 1 Average Magnitude: 20.397287764777772
Lens Average Magnitude: 21.57967985869685
Extended Source Average Magnitude: 21.391098277803636
Intrinsic Source Average Magnitude: 25.04769601095083
